# RBD analysis — Step 8 Visualization

Two panels:
1. **Example proteins**: per-residue cross-attention along a chosen RBP, with RBD intervals shaded.
2. **Aggregate**: distribution of per-pair `mean_diff` across all pairs, colored by `dominant_rbd`.

Inputs (produced by the RBD_analysis pipeline):
- `results/<model>/rbd_attention_stats.csv` (per-pair stats)
- `results/<model>/rbd_pair_vectors.npz` (cached attention + mask)
- `results/<model>/rbd_annotations.rbd_intervals.tsv` (interval coordinates)

Run from the repo root.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repo-root bootstrap: makes `coral` importable without installing the package.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from coral import paths
from coral.rbd import load_rbd_intervals

RES = paths.analysis_results("RBD_analysis")
HEADS = RES / "heads_best"
print("results dir:", RES)

In [ ]:
stats_df = pd.read_csv(HEADS / "rbd_attention_stats.csv")
intervals = load_rbd_intervals(RES / "rbd_annotations.rbd_intervals.tsv")
data = np.load(HEADS / "rbd_protein_vectors.npz", allow_pickle=True)
unit_ids = list(data["unit_ids"])
protein_ids = list(data["protein_ids"])
attentions = list(data["attentions"])
masks = list(data["masks"])
print(f"{len(stats_df)} per-protein rows, {len(unit_ids)} cached vectors")

## Panel 1 — Example proteins

Pick proteins with well-known multi-RBD architecture. Default candidates below are UniProt accessions; if none are present, fall back to the protein with the largest per-pair `mean_diff` and ≥ 2 merged RBD intervals.

In [ ]:
candidates = ["Q15717", "P26378", "Q9NZI8", "O00425"]  # ELAVL1, ELAVL4, IGF2BP1, IGF2BP3

available = [p for p in candidates if p in protein_ids]
if not available:
    multi_rbd = [pid for pid, ivs in intervals.items() if len(ivs) >= 2]
    candidates_fb = (stats_df[stats_df["protein_id"].isin(multi_rbd)]
                     .sort_values("mean_diff", ascending=False)
                     .drop_duplicates("protein_id")
                     .head(2)["protein_id"].tolist())
    available = candidates_fb
print("plotting:", available)

In [ ]:
def plot_protein_attention(ax, protein_id):
    """Mean attention vector across this protein's pairs, with RBD intervals shaded."""
    idxs = [i for i, p in enumerate(protein_ids) if p == protein_id]
    if not idxs:
        ax.set_visible(False)
        return
    L = max(attentions[i].shape[0] for i in idxs)
    stack = np.full((len(idxs), L), np.nan, dtype=np.float32)
    for k, i in enumerate(idxs):
        stack[k, :attentions[i].shape[0]] = attentions[i]
    mean_attn = np.nanmean(stack, axis=0)

    ax.plot(np.arange(L), mean_attn, lw=1.0, color="black")
    for s, e in intervals.get(protein_id, []):
        ax.axvspan(s - 1, e, color="tab:orange", alpha=0.25)
    ax.set_title(f"{protein_id}  ({len(idxs)} RNA partners)")
    ax.set_xlabel("residue")
    ax.set_ylabel("mean cross-attention")

if available:
    fig, axes = plt.subplots(len(available), 1, figsize=(10, 2.4 * len(available)),
                              squeeze=False)
    for ax, pid in zip(axes[:, 0], available):
        plot_protein_attention(ax, pid)
    fig.tight_layout()
    plt.show()
else:
    print("No example proteins available in this model run.")

## Panel 2 — Aggregate per-pair Δ by dominant RBD family

In [ ]:
fam_counts = stats_df["dominant_rbd"].value_counts()
keep_fams = fam_counts[fam_counts >= 20].index.tolist()
plot_df = stats_df[stats_df["dominant_rbd"].isin(keep_fams)].copy()
order = (plot_df.groupby("dominant_rbd")["mean_diff"].median()
         .sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(max(6, 0.6 * len(order)), 4.5))
data_per_fam = [plot_df.loc[plot_df["dominant_rbd"] == f, "mean_diff"].values
                for f in order]
ax.boxplot(data_per_fam, tick_labels=order, showfliers=False)
ax.axhline(0, color="red", lw=0.8, ls="--")
ax.set_ylabel("per-pair Δ  =  mean(attn|RBD) − mean(attn|¬RBD)")
ax.set_xlabel("dominant RBD family")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

print(f"families with ≥20 pairs: {len(order)}; total pairs plotted: {len(plot_df)}")
print(plot_df.groupby("dominant_rbd")["mean_diff"].agg(["count", "mean", "median"]).loc[order])

## Panel 3 — Real Δ̄ vs position-shuffle null (if `controls.py` has been run)

In [ ]:
null_path = HEADS / "rbd_null_distribution.csv"
if null_path.exists():
    null_df = pd.read_csv(null_path)
    observed = float(stats_df["mean_diff"].mean())
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(null_df["shuffled_mean_diff"], bins=40, color="#888", alpha=0.85)
    ax.axvline(observed, color="red", lw=2, label=f"observed Δ̄ = {observed:.2e}")
    ax.set_xlabel("aggregate Δ̄ over proteins")
    ax.set_ylabel("shuffle iterations")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print(f"{null_path} not found — run controls.py to generate it.")